In [1]:
#Code for importing lib
!pip install ultralytics
# Step 1: Import necessary libraries
import os
import cv2
import matplotlib.pyplot as plt
from google.colab import drive
from ultralytics import YOLO

# Step 2: Mount Google Drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


In [2]:
#Code for getting performace metrix

import os
import pandas as pd

# Define the path to your validation results
results_path = '/content/drive/MyDrive/Colab Notebooks/Final_training_garbage_96_percent/exp'

# Load the results CSV file
results_file = os.path.join(results_path, 'results.csv')

if os.path.exists(results_file):
    # Read the results
    results_df = pd.read_csv(results_file)

    # Print the available columns to debug
    print("Available columns in results file:")
    print(results_df.columns)

    # Extract metrics using the correct column names
    if 'metrics/mAP50(B)' in results_df.columns and 'metrics/mAP50-95(B)' in results_df.columns:
        mAP50 = results_df['metrics/mAP50(B)'].values[-1]  # mAP at IoU=0.5
        mAP50_95 = results_df['metrics/mAP50-95(B)'].values[-1]  # mAP at IoU from 0.5 to 0.95
        print(f"mAP@0.5: {mAP50}, mAP@0.5:0.95: {mAP50_95}")
    else:
        print("mAP columns not found in the results file.")

    if 'metrics/precision(B)' in results_df.columns and 'metrics/recall(B)' in results_df.columns:
        precision = results_df['metrics/precision(B)'].values[-1]  # Precision
        recall = results_df['metrics/recall(B)'].values[-1]  # Recall
        print(f"Precision: {precision}, Recall: {recall}")
    else:
        print("Precision and Recall columns not found in the results file.")

else:
    print("Results file not found. Please check the results directory.")


Available columns in results file:
Index(['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
       'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)',
       'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss',
       'lr/pg0', 'lr/pg1', 'lr/pg2'],
      dtype='object')
mAP@0.5: 0.96986, mAP@0.5:0.95: 0.87729
Precision: 0.96789, Recall: 0.94189


In [3]:
!pip install ultralytics
# Step 1: Import necessary libraries
import os
import cv2
import matplotlib.pyplot as plt
from google.colab import drive
from ultralytics import YOLO

# Step 2: Mount Google Drive
drive.mount('/content/drive')

# Step 3: Unzip the dataset
zip_file_path = '/content/drive/MyDrive/Colab Notebooks/garbage_detection.v1i.yolov8.zip'  # Replace with your zip file path
output_folder = '/content/dataset'  # Define an output folder

# Unzipping the dataset
!unzip -qo "{zip_file_path}" -d "{output_folder}"

# Step 4: Load the trained YOLOv8 model
model_path = '/content/drive/MyDrive/Colab Notebooks/Final_training_garbage_96_percent/exp/weights/best.pt'  # Replace with your trained model path
model = YOLO(model_path)

# Step 5: Function to classify a single image
def classify_image(image_path):
    # Read the image
    img = cv2.imread(image_path)

    # Convert BGR to RGB for YOLO model compatibility
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Perform inference
    results = model.predict(img)

    # Extract results
    detection_results = results[0]  # Get the first result if multiple images were processed

    # Display the image with bounding boxes
    plt.imshow(detection_results.plot())
    plt.axis('off')
    plt.show()

    # Create a DataFrame for results
    boxes = detection_results.boxes
    if boxes is not None:
        data = {
            "x1": boxes.xyxy[:, 0].tolist(),
            "y1": boxes.xyxy[:, 1].tolist(),
            "x2": boxes.xyxy[:, 2].tolist(),
            "y2": boxes.xyxy[:, 3].tolist(),
            "confidence": boxes.conf.tolist(),
            "class": boxes.cls.tolist(),
        }
        return data  # Return data as a dictionary
    else:
        print("No detections found.")
        return None

# Step 6: Main script for classification
while True:
    choice = input("Enter '1' to classify specific images or '2' to classify all images in the Test images folder (or 'q' to quit): ")

    if choice == '1':
        num_images = int(input("How many images do you want to enter? "))
        for _ in range(num_images):
            image_name = input("Enter the image name (including extension): ")
            image_path = os.path.join(output_folder, 'test/images', image_name)  # Adjust folder path as needed
            if os.path.exists(image_path):
                results = classify_image(image_path)
                print(f"Results for {image_name}:\n", results)
            else:
                print(f"Image '{image_name}' not found. Please check the file name and try again.")

    elif choice == '2':
        test_images_folder = os.path.join(output_folder, 'test/images')  # Adjust folder path as needed
        for image_name in os.listdir(test_images_folder):
            if image_name.endswith(('.jpg', '.jpeg', '.png')):  # Adjust extensions as needed
                image_path = os.path.join(test_images_folder, image_name)
                results = classify_image(image_path)
                print(f"Results for {image_name}:\n", results)

    elif choice.lower() == 'q':
        print("Exiting the program.")
        break

    else:
        print("Invalid choice. Please try again.")


Output hidden; open in https://colab.research.google.com to view.